In [29]:
from dotenv import load_dotenv
import os 
load_dotenv()

from openai import OpenAI

In [30]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [31]:
# our black box: iput = text, output = text 
def llm(prompt):
    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=prompt
    )
    return response.output_text

In [32]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

You'd like to join the course. I need a bit more information from you. Could you please provide more context or details about the course you're referring to? What type of course is it, and is it being offered online or in-person? Additionally, what's the current enrollment status or deadline for joining the course? This will help me better understand your situation and provide a more accurate response.


In [33]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [34]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.
Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question: 
{question}

Context: 
{context}
"""

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In [35]:
# this is our goal (MAIN ARCHITECTURE): 3 steps => search, prompt, send result to LLM
def rag(question): 
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [36]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [37]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 84},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [38]:
documents = []
courses_rawdocuments = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() # if something is broken raise an issue, dont continue
    course_data = course_response.json()

    documents.extend(course_data) # put everything in one list 

len(documents) 

1349

In [39]:
documents[1100]

{'id': '84301e35dd',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': "ConnectionError 'Connection aborted.' for --bind 127.0.0.1:5000",
 'answer': 'I was getting an error on the client side with this:\n\n**Client Side Error:**\n\n```plaintext\nFile "C:\\python\\lib\\site-packages\\urllib3\\connectionpool.py", line 703, in urlopen ...\n\nraise ConnectionError(err, request=request)\n\nrequests.exceptions.ConnectionError: (\'Connection aborted.\', RemoteDisconnected(\'Remote end closed connection without response\'))\n```\n\n**Server Side:**\n\nAn error was shown for Gunicorn, although the `waitress` command was running smoothly from the server side.\n\n**Solution:**\n\n- Use the IP address `0.0.0.0:8000` or `0.0.0.0:9696`. They are the ones which work most of the time.'}

In [40]:
# generally we need a lot of time working on the data, parsing cleaning etc.. 
# we will now take FAQ data and index it in a way that its possible to send queries to perform search on this data  
# search engine ! 

# Query: "how to install python"
# will match any doc that mentions "how", "install", or "python"
# the more words match → the higher the score → ranked higher

In [41]:
from minsearch import Index

# Step 1: define the structure (what fields, what type)
index = Index(
    text_fields=["question", "section", "answer"], # will match answers mentioning a word in our question 
    keyword_fields=["course"] # keyword we want to filter by 
)

# Step 2: load the actual data
index.fit(documents)

In [42]:
# search with filters and boosts 
search_results = index.search(question, 
                              boost_dict={"question":2.0,  # give more weight to certain fields
                                          "section": 0.5},
             filter_dict={'course': "llm-zoomcamp"}, # hard filter, only search within this course
             num_results=5) # return top 5 most relevant docs

In [43]:
# we can boost records, like one field is more interesting/important than the other. 
# if its below 1 then its less important

In [44]:
# in one function 
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [45]:
question = "how do i start this course ?"

search_results = search(question)

In [46]:
for doc in search_results: 
    print(doc["question"])

How should I start the course and follow the weekly workflow?
OpenAI: Do I have to subscribe and pay for Open AI API for this course?
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?
Certificate: Can I follow the course in a self-paced mode and get a certificate?


In [57]:
question = "can i still join? "
search_results = search(question)

In [58]:
# BUILDING THE PROMPT
# instructions tell the LLM its role and how to answer 

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""

# user prompt has placeholders for the question and context 
USER_PROMPT_TEMPLATE = """

Question: 
{question}

Context: 
{context}

"""

In [59]:
# building the context; parsing 
def build_context(search_results): 

    lines =[]
    for doc in search_results: 
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return  "\n".join(lines).strip()

In [60]:
def build_prompt(question, search_results):
    
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question, 
        context = context
    )
    return prompt.strip()

In [61]:
prompt = build_prompt(question, search_results)

In [62]:
print(prompt)

Question: 
can i still join? 

Context: 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

Module 1 Homework
Q: Where can I find the homework questions?
A: Homework links are available in the course GitHub repo and in the course management platform.

For the 2026 Module 1 homework, use:

- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)
- [Module 1 homework instructions](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homewo

In [63]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=prompt
    )

In [64]:
response.output_text

'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, note that while homework is not mandatory, it is recommended for reinforcing concepts and the points awarded count towards your rank on the leaderboard. You can find the homework questions and instructions on the course GitHub repo and the course management platform.'

In [67]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01kvjcege3ep7a7w7sw0etxph5",
  "created_at": 1781954724.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "llama-3.3-70b-versatile",
  "object": "response",
  "output": [
    {
      "id": "resp_01kvjcege3ep7s2m1qs1q51rz5",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_01kvjcege3ep88m41q9qz3gz25",
      "content": [
        {
          "annotations": [],
          "text": "Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, note that while homework is not mandatory, it is recommended for reinforcing concepts and the points awarded count towards your rank on the leaderboard. You can find the homework questions and instructions on the course GitHub repo and the course management platform.",
     

In [74]:
response.usage

ResponseUsage(input_tokens=595, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=78, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=673)

In [75]:
response.output[0] # the message

ResponseReasoningItem(id='resp_01kvjcege3ep7s2m1qs1q51rz5', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')

In [77]:
response.output_text # short cut 

'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, note that while homework is not mandatory, it is recommended for reinforcing concepts and the points awarded count towards your rank on the leaderboard. You can find the homework questions and instructions on the course GitHub repo and the course management platform.'

In [78]:
# calcualting the price of a query: this is an example using gpt not llama
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00079725

In [83]:
def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
            
    api_input = [
        {"role": "developer", "content": instructions}, 
        {"role": "user", "content": user_prompt}

    ]
    response = openai_client.responses.create(
        model=model,
        input=api_input 
        )
    return response.output_text

open ai has two types of APIs, chat compeletions and responses. responses is newer.  

system prompt inside gpt that we don't see, instruction. 
then we send our user prompt, and then conversation history. 

In [84]:
# full RAG
def rag(query, model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [85]:
answer = rag(question)
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted. Additionally, while homework is not mandatory, it's recommended for reinforcing concepts, and you'll need to pass the Capstone project to get the certificate.


In [87]:
rag("How do I get a certificate?")

'To get a certificate, you need to follow these steps:\n\n1. **Complete the course with a "live" cohort**: You cannot get a certificate if you\'re taking the course in self-paced mode. You need to finish the course with a live cohort to be eligible for a certificate.\n2. **Peer-review 3 capstone projects**: After submitting your project, you need to peer-review 3 capstone projects from your cohort members. This can only be done during the time the course is running.\n3. **Pass the Capstone project**: You need to pass the Capstone project to get the certificate. Homework is not mandatory, but it\'s recommended to reinforce concepts.\n4. **Ensure your official name is correctly listed**: Make sure your official name (as it appears on your identification documents) is correctly listed in your course profile, as this will be the name that appears on your certificate.\n\nBy following these steps, you can obtain a certificate upon completing the course.'

In [91]:
print(rag("is this about the final project ? what do you mean by capstone project ? "))

The text is indeed about the capstone project, which appears to be a final project for a course or certification program. 

A capstone project is a culminating project that requires students to apply the knowledge and skills they have acquired throughout the course to a real-world problem or scenario. It is typically an individual project, although students may be allowed to collaborate or discuss ideas with others. 

In this specific case, the capstone project involves submitting a project that will be evaluated by peer reviewers, and students are also required to review and grade the projects of their peers. The evaluation criteria and guidelines are provided, and students are expected to follow them in order to receive a certificate at the end of the course.

The capstone project has the following key aspects:

1. **Individual project**: Each student must submit their own project, although they can collaborate or discuss ideas with others.
2. **Peer review**: Students will review an

In [94]:
print(rag("do i need to submit my homework before the deadline ? "))

According to the provided context, the answer to your question "Do I need to submit my homework before the deadline?" is implied to be yes. 

Although the text doesn't directly answer your question, it does mention "deadlines" in the context of the course platform being useful for submission and deadlines. It also points to "[homework logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/homework/)" for general homework guidance, which likely includes information about submission deadlines.

Therefore, it's reasonable to assume that submitting homework before the deadline is a requirement, but for explicit confirmation, you might want to check the specific homework instructions or the course platform.


In [ ]:
from toyaikit import Agent


def search(query: str) -> list:
    """Search the course FAQ chunks for relevant information."""
    return index.search(query, num_results=5)


instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

agent = Agent(
    client=openai_client,
    model="zai-glm-4.7",  # or whatever model you're using
    instructions=instructions,
    tools=[search])


result = agent.run("How does the agentic loop work, and how is it different from plain RAG?")
print(result)

ImportError: cannot import name 'Agent' from 'toyaikit' (/Users/faten/Desktop/rag-llm/RAG_project/.venv/lib/python3.13/site-packages/toyaikit/__init__.py)